# Model C_4: RoBERTa (model umum) vs FinBERT (model keuangan)

**Pertanyaan.** Apakah model yang dilatih khusus teks keuangan (FinBERT, dipakai di C_2) memang lebih baik daripada model bahasa umum (RoBERTa) untuk memprediksi kurs dari berita? Ini contoh "domain vs general embeddings" di soal (1b-iv).

**Cara membandingkannya adil.** Semuanya dibuat SAMA seperti C_2, hanya modelnya yang ganti:
- artikel Inggris yang sama (corpus B_2), tiap artikel = "judul. paragraf pembuka" (judul dan lead);
- tiap artikel diubah menjadi 768 angka oleh model, lalu dirata-rata per hari;
- PCA 20 komponen (dilatih hanya di hari train) + penanda `has` (1 kalau hari itu ada berita);
- model LR dan HGB dengan harga + fitur teks.

**Apa bedanya FinBERT dan RoBERTa?** Keduanya "pembaca teks" yang mengubah kalimat jadi angka. FinBERT dilatih dengan banyak teks keuangan, jadi mungkin lebih paham istilah seperti "rate hike". RoBERTa dilatih dengan teks umum (buku, berita, web).

**Pooling: dua cara, sama seperti C_2.** "Pooling" = cara merangkum satu artikel (banyak token) jadi satu vektor. `cls` = ambil vektor token pertama; `mean` = rata-rata semua token. Untuk RoBERTa, token pertama tidak dilatih untuk merangkum kalimat, jadi `mean` dipilih sebagai yang utama SEBELUM melihat hasil; `cls` ikut dijalankan agar sebanding dengan C_2.

**Empat varian:** 2 pooling x 2 model (LR, HGB). Pembanding yang adil adalah FinBERT dengan pooling yang sama (`C2_finbert_art_mean_*` dan `C2_finbert_art_cls_*`).

In [1]:
from c_common import *
import json
import torch
from sklearn.decomposition import PCA

D, text = load_all()   # corpus B_2: hanya artikel Inggris terfilter
print({s: len(d) for s, d in D.items()}, text.shape)

{'train': 825, 'val': 176, 'test': 178} (1202, 5)


## 1. Pengaturan

In [2]:
MODEL_NAME = "FacebookAI/roberta-base"   # diunduh sekali (sekitar 0,5 GB) ke cache Hugging Face bawaan (drive C:)
POOLS = ("mean", "cls")        # dua cara merangkum artikel, sama seperti C_2. "mean" dipilih sebagai yang utama sebelum melihat hasil.
N_COMP = 20                    # jumlah komponen PCA (sama dengan C_2)
CHUNK = 510                    # 512 dikurangi 2 token khusus di awal dan akhir
BATCH = 16
LIMIT = None                   # angka kecil (mis. 50) = hanya 50 hari pertama, untuk uji cepat (tidak disimpan). None = semua.

f_cls = FEATURES_C / "c4_roberta_cls.npy"
f_mean = FEATURES_C / "c4_roberta_mean.npy"
TITLE, LEAD = "judul_gabungan", "lead_paragraph_gabungan"     # kolom corpus B_2 (sama dengan C_2)

## 2. Muat RoBERTa

Pertama kali dijalankan, model diunduh (sekitar 0,5 GB). Setelah itu dibaca dari cache.

In [3]:
from transformers import AutoTokenizer, AutoModel

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).eval()
print(MODEL_NAME, "| parameter:", round(sum(p.numel() for p in model.parameters()) / 1e6), "juta")

W1120 16:36:16.656000 13800 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: FacebookAI/roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


FacebookAI/roberta-base | parameter: 125 juta


## 3. Satu baris per artikel: "judul. lead"

Sama persis dengan C_2: judul tiap artikel digabung dengan paragraf pembukanya, lalu diubah menjadi nomor-nomor kata (token) dan dipotong di 510 token. Judul dan lead tiap hari dipisah dengan `" | "`; kalau jumlahnya tidak cocok di suatu hari, hari itu hanya memakai judul.

In [4]:
def build_article_rows(text_df, tok):
    """Satu baris per artikel: "judul. lead" (dipotong 510 token). Return list (posisi_hari, id_artikel, token_ids)."""
    rows, art_id, fallback = [], 0, 0
    for pos, (t, l) in enumerate(zip(text_df[TITLE].fillna(""), text_df[LEAD].fillna(""))):
        titles = [x.strip() for x in t.split(" | ") if x.strip()]
        leads = [x.strip() for x in l.split(" | ")]
        if len(leads) != len(titles):          # jumlah tidak cocok -> judul saja
            leads, fallback = [""] * len(titles), fallback + 1
        for ti, li in zip(titles, leads):
            ids = tok(f"{ti}. {li}".strip(" ."), add_special_tokens=False, truncation=True, max_length=CHUNK)["input_ids"]
            rows.append((pos, art_id, ids))
            art_id += 1
    print("hari memakai judul saja (jumlah judul/lead tidak cocok):", fallback)
    return rows


has_en = (text["jumlah_berita"] > 0).values
print("hari dengan artikel:", int(has_en.sum()), "dari", len(text), "| artikel:", int(text["jumlah_berita"].sum()))

hari dengan artikel: 1182 dari 1202 | artikel: 11764


## 4. Ubah tiap artikel menjadi 768 angka, lalu rata-ratakan per hari

- Artikel diurutkan menurut panjang supaya "padding" (pengisi supaya semua teks dalam satu batch sama panjang) seminimal mungkin. Padding diabaikan saat menghitung `mean`.
- Hasil disimpan ke `data/features_c/c4_roberta_{cls,mean}.npy`, jadi pembuatan embedding hanya sekali. Di GPU sekitar beberapa menit.

In [5]:
@torch.no_grad()
def embed_chunks(rows, tok, model, batch=BATCH):
    """RoBERTa untuk semua artikel. Return (CLS, mean) masing-masing (n_artikel, 768)."""
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", dev)
    model.to(dev).eval()
    order = np.argsort([len(r[2]) for r in rows])        # urut menurut panjang -> padding minimal
    cls = np.zeros((len(rows), 768), dtype="float32")
    mean = np.zeros_like(cls)
    for s in range(0, len(order), batch):
        ix = order[s:s + batch]
        # RoBERTa memakai token khusus <s> di awal dan </s> di akhir (cls_token_id dan sep_token_id)
        enc = tok.pad({"input_ids": [[tok.cls_token_id] + list(rows[i][2]) + [tok.sep_token_id] for i in ix]}, return_tensors="pt")
        enc = {k: v.to(dev) for k, v in enc.items()}
        h = model(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1)
        cls[ix] = h[:, 0].float().cpu().numpy()
        mean[ix] = ((h * m).sum(1) / m.sum(1)).float().cpu().numpy()   # padding diabaikan
        if (s // batch) % 200 == 0:
            print(f"  {s}/{len(order)} artikel")
    model.cpu()
    return cls, mean


def pool_to_days(rows, vecs, n_days):
    """Rata-rata artikel -> hari. Hari tanpa artikel = vektor nol."""
    df = pd.DataFrame(vecs)
    df["day"] = [r[0] for r in rows]
    day_vec = df.groupby("day").mean()
    out = np.zeros((n_days, vecs.shape[1]), dtype="float32")
    out[day_vec.index.values] = day_vec.values
    return out


FEATURES_C.mkdir(parents=True, exist_ok=True)
if LIMIT is None and f_cls.exists() and f_mean.exists():
    emb = {"cls": np.load(f_cls), "mean": np.load(f_mean)}
    print("dimuat dari cache:", emb["cls"].shape)
else:
    rows = build_article_rows(text, tok)
    if LIMIT:
        keep = set(np.where(has_en)[0][:LIMIT])
        rows = [r for r in rows if r[0] in keep]
    print(len(rows), "artikel")
    c, m = embed_chunks(rows, tok, model)
    emb = {"cls": pool_to_days(rows, c, len(text)), "mean": pool_to_days(rows, m, len(text))}
    if LIMIT is None:                                    # jangan simpan cache kalau hanya uji sebagian
        np.save(f_cls, emb["cls"]); np.save(f_mean, emb["mean"])
        print("disimpan ke", FEATURES_C)
print("NaN:", {p: bool(np.isnan(emb[p]).any()) for p in POOLS}, "| hari dengan vektor:", int((np.abs(emb["mean"]).sum(1) > 0).sum()))

dimuat dari cache: (1202, 768)
NaN: {'mean': False, 'cls': False} | hari dengan vektor: 1182


## 5. Kecilkan menjadi 20 angka (PCA) dan gabungkan ke data

768 angka terlalu banyak untuk 825 hari latihan, jadi diringkas menjadi 20 angka dengan PCA. PCA hanya "belajar" dari hari train yang punya berita (tidak mengintip validation/test). Hari tanpa berita diberi 0 di semua kolom dan `has = 0`.

In [6]:
def add_pca_features(emb, prefix, has):
    """PCA di-fit HANYA pada hari train yang punya teks, lalu diterapkan ke semua hari. Fitur digabung ke D lewat tanggal."""
    has = np.asarray(has, bool)
    feats = pd.DataFrame(emb, index=text.index)
    tr = D["train"]["date"]
    tr = tr[has[text.index.get_indexer(tr)]]
    pca = PCA(n_components=N_COMP, random_state=SEED).fit(feats.loc[tr].values)
    z = pca.transform(feats.values)
    z[~has] = 0.0
    comp = pd.DataFrame(z, index=text.index, columns=[f"{prefix}_{i}" for i in range(N_COMP)])
    comp[f"{prefix}_has"] = has.astype(float)
    for s in D:
        D[s] = D[s].drop(columns=[c for c in comp.columns if c in D[s].columns]) \
                   .merge(comp, left_on="date", right_index=True, how="left")
    print(prefix, "variansi dijelaskan PCA:", round(pca.explained_variance_ratio_.sum(), 3))
    return list(comp.columns)


RB = {p: add_pca_features(emb[p], f"c4_roberta_{p}", has_en) for p in POOLS}

c4_roberta_mean variansi dijelaskan PCA: 0.714
c4_roberta_cls variansi dijelaskan PCA: 0.644


## 6. Nilai di validation

Empat varian: 2 cara pooling (`mean`, `cls`) x 2 model (LR, HGB), harga + fitur RoBERTa. Skor utama untuk dibaca: **AUC** (0,5 = sama dengan menebak acak; makin dekat 1 makin baik). Laporkan jumlah konfigurasi: C_1: 2, C_2: 6, C_3: 4, C_4: 4.

In [7]:
VARIANTS = {
    f"C4_roberta_{p}_{k}": (k, RB[p], p) for p in POOLS for k in ("LR", "HGB")
}
for name, (kind, cols, p) in VARIANTS.items():
    evaluate_val(name, kind, D, PRICE + cols, n_configs_tried=len(VARIANTS),
                 note=f"RoBERTa-base {p}, judul+lead per artikel, PCA {N_COMP}")

C4_roberta_mean_LR [LR] val: DA=0.494, BalAcc=0.499, F1_macro=0.487, MCC=-0.002, AUC=0.488


C4_roberta_mean_HGB [HGB] val: DA=0.562, BalAcc=0.564, F1_macro=0.562, MCC=0.128, AUC=0.596
C4_roberta_cls_LR [LR] val: DA=0.472, BalAcc=0.475, F1_macro=0.469, MCC=-0.052, AUC=0.499


C4_roberta_cls_HGB [HGB] val: DA=0.483, BalAcc=0.488, F1_macro=0.472, MCC=-0.025, AUC=0.488


## 7. Bandingkan (validation)

- **harga saja**: model tanpa teks sama sekali. Teks baru berguna kalau skornya jelas di atasnya.
- **C2_finbert_art_mean_*** dan **C2_finbert_art_cls_***: FinBERT dengan pooling yang sama dengan RoBERTa (bandingkan `mean` dengan `mean`, `cls` dengan `cls`).

Perbedaan AUC sekitar 0,04 atau kurang dengan 176 hari validation bisa muncul hanya karena kebetulan. Jadi hanya anggap satu model "lebih baik" kalau selisihnya jelas lebih besar.

In [8]:
rows = []
for k in ["LR", "HGB"]:
    _, sc, _ = tune(k, D, PRICE)
    rows.append({"model": f"harga saja {k}", **sc})
for name in ["C2_finbert_art_mean_HGB", "C2_finbert_art_cls_HGB"] + list(VARIANTS):
    f = RESULTS_C / f"{name}.json"
    if f.exists():
        rows.append({"model": name, **json.loads(f.read_text())["val"]})
    else:
        print("belum ada:", name)
pd.DataFrame(rows).round(3)

,model,DA,BalAcc,F1_macro,MCC,AUC
0,harga saja LR,0.506,0.508,0.504,0.016,0.501
1,harga saja HGB,0.523,0.521,0.520,0.042,0.503
2,C2_finbert_art_mean_HGB,0.489,0.489,0.489,-0.022,0.458
3,C2_finbert_art_cls_HGB,0.591,0.588,0.588,0.179,0.568
4,C4_roberta_mean_LR,0.494,0.499,0.487,-0.002,0.488
5,C4_roberta_mean_HGB,0.562,0.564,0.562,0.128,0.596
6,C4_roberta_cls_LR,0.472,0.475,0.469,-0.052,0.499
7,C4_roberta_cls_HGB,0.483,0.488,0.472,-0.025,0.488
